# Convolutional Neural Networks (CNN): Operations, Receptive Fields & Architectures

This interactive lab covers:
1. **2D Convolution & Pooling from Scratch**: Mathematical verification against PyTorch.
2. **Receptive Field Dynamics**: Tracking effective sensory field growth through convolutional hierarchies.
3. **Residual Learning (ResNet)**: Identity shortcuts and gradient highway mechanics.
4. **Mobile & Efficient Architectures (MBConv)**: Depthwise separable convolutions + Squeeze-and-Excitation.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from cnn_engine import (
    conv2d_forward_scratch, maxpool2d_forward_scratch,
    compute_receptive_field, LeNet5, ResidualBlock, MiniResNet, MBConvBlock
)

print('CNN Engine modules loaded successfully!')

## 1. Scratch 2D Convolution vs. PyTorch Reference

We test our manual sliding-window convolution on an input feature map and compare directly with `torch.nn.functional.conv2d`.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

X_np = np.random.randn(2, 3, 8, 8).astype(np.float32)
W_np = np.random.randn(4, 3, 3, 3).astype(np.float32)
b_np = np.random.randn(4).astype(np.float32)

stride = 1
padding = 1
out_scratch = conv2d_forward_scratch(X_np, W_np, b_np, stride=stride, padding=padding)
out_pt = F.conv2d(torch.tensor(X_np), torch.tensor(W_np), torch.tensor(b_np), stride=stride, padding=padding).numpy()

diff = np.max(np.abs(out_scratch - out_pt))
print(f'Output feature map shape: {out_scratch.shape}')
print(f'Max absolute difference vs. PyTorch: {diff:.2e}')
assert np.allclose(out_scratch, out_pt, atol=1e-5), 'Scratch Conv2D does not match PyTorch!'

## 2. Receptive Field Progression Analysis

We compute how the effective receptive field grows as we stack $3 \times 3$ convolutions versus downsampling with stride.

In [ ]:
conv_stack = [
    {'k': 3, 's': 1, 'name': 'Conv 3x3 (s=1)'},
    {'k': 3, 's': 1, 'name': 'Conv 3x3 (s=1)'},
    {'k': 2, 's': 2, 'name': 'MaxPool 2x2 (s=2)'},
    {'k': 3, 's': 1, 'name': 'Conv 3x3 (s=1)'},
    {'k': 3, 's': 1, 'name': 'Conv 3x3 (s=1)'},
]

rf_history = compute_receptive_field(conv_stack)
print(f'{"Layer":<22} | {"Kernel":<6} | {"Stride":<6} | {"Receptive Field":<16} | {"Jump Factor":<10}')
print('-' * 70)
for layer_cfg, res in zip(conv_stack, rf_history):
    print(f"{layer_cfg['name']:<22} | {res['kernel']:<6} | {res['stride']:<6} | {res['receptive_field']:<16} | {res['jump']:<10}")

## 3. MiniResNet Architecture & Forward Flow

We instantiate our `MiniResNet` architecture with residual identity blocks and evaluate parameter counts and output predictions.

In [ ]:
model = MiniResNet(in_channels=3, num_classes=10)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

x_batch = torch.randn(8, 3, 32, 32)
logits = model(x_batch)

print(f'MiniResNet Total Trainable Parameters: {total_params:,}')
print(f'Input Batch Shape:  {x_batch.shape}')
print(f'Output Logits Shape: {logits.shape}')
assert logits.shape == (8, 10), 'Invalid output logits shape!'

## 4. Modern Efficient Architectures: MBConv Block (EfficientNet)

We verify the forward activation of an inverted residual bottleneck with depthwise separable convolution and Squeeze-and-Excitation channel attention.

In [ ]:
mbconv = MBConvBlock(in_channels=32, out_channels=32, expand_ratio=6, stride=1, se_ratio=0.25)
x_mb = torch.randn(4, 32, 28, 28)
out_mb = mbconv(x_mb)

print(f'MBConv Input Shape:  {x_mb.shape}')
print(f'MBConv Output Shape: {out_mb.shape}')
assert out_mb.shape == x_mb.shape, 'MBConv with stride=1 must preserve spatial dimensions!'